<div align="center">

<img src="../images/Logo-Uni-Osnabrueck.jpg" width="300"/>

# Introduction to Computational Linguistics

</div>

## Regular expressions

### Cheatsheet

| Pattern | Meaning | Example match |
|---------|---------|---------------|
| `.` | Any character (except newline) | `c.t` → "cat", "cut", "c3t" |
| `*` | 0 or more of previous | `go*d` → "gd", "god", "good" |
| `+` | 1 or more of previous | `go+d` → "god", "good" (not "gd") |
| `?` | 0 or 1 of previous | `colou?r` → "color", "colour" |
| `^` | Start of string | `^Hello` → "Hello world" |
| `$` | End of string | `world$` → "Hello world" |
| `[abc]` | Any one of a, b, c | `[aeiou]` → any vowel |
| `[^abc]` | Any char NOT in set | `[^aeiou]` → any consonant |
| `\w` | Word character `[a-zA-Z0-9_]` | `\w+` → any word |
| `\d` | Digit `[0-9]` | `\d+` → "42", "2026" |
| `\s` | Whitespace | `\s+` → spaces, tabs |
| `\b` | Word boundary | `\bcat\b` → "cat" not "catch" |
| `(abc)` | Capture group | `(go)+` → "gogo" |
| `a\|b` | Either a or b | `cat\|dog` → "cat" or "dog" |
| `{n,m}` | Between n and m repetitions | `\d{2,4}` → "42", "2026" |

### Regex in Bash — `grep`

`grep` searches for lines matching a pattern.

```bash
grep 'pattern' file.txt        # basic search
grep -E 'pattern' file.txt     # extended regex (ERE)
grep -o 'pattern' file.txt     # print only the matched part
grep -i 'pattern' file.txt     # case-insensitive
```

In [1]:
%%bash
TEXT="The cat sat on the mat. A cat in a hat. 2 cats, 3 dogs."

# Lines containing "cat"
echo "$TEXT" | grep -o 'cat'

# Words starting with capital letter
echo "$TEXT" | grep -oE '\b[A-Z][a-z]+'

# All numbers
echo "$TEXT" | grep -oE '\d+' || echo "$TEXT" | grep -oE '[0-9]+'

# Words ending in "at"
echo "$TEXT" | grep -oE '\b\w+at\b'

cat
cat
cat
The
2
3
cat
sat
mat
cat
hat


### Regex in Python — `re` module

The four functions you'll use most:

| Function | What it does |
|----------|-------------|
| `re.search(pattern, text)` | Find first match anywhere in text |
| `re.match(pattern, text)` | Match only at the **start** of text |
| `re.findall(pattern, text)` | Return **all** matches as a list |
| `re.sub(pattern, replacement, text)` | **Replace** matches |

In [2]:
import re

text = "The cat sat on the mat. A cat in a hat. 2 cats, 3 dogs."

# findall() returns a list of ALL matches
words_ending_at = re.findall(r'\b\w+at\b', text)
print("Words ending in 'at':", words_ending_at)

numbers = re.findall(r'\d+', text)
print("Numbers found:", numbers)

# search() finds only the FIRST match and returns a match object
match = re.search(r'\bcat\b', text)
if match:
    print("First 'cat' at position", match.start(), "to", match.end())

# sub() REPLACES every match with a new string
censored = re.sub(r'\bcat\b', '***', text)
print("Censored:", censored)

# Parentheses () create groups — each group is captured separately
date_text = "Published on 2026-05-12 and updated on 2026-05-15"
dates = re.findall(r'(\d{4})-(\d{2})-(\d{2})', date_text)
print("Dates found (year, month, day):", dates)

Words ending in 'at': ['cat', 'sat', 'mat', 'cat', 'hat']
Numbers found: ['2', '3']
First 'cat' at position 4 to 7
Censored: The *** sat on the mat. A *** in a hat. 2 cats, 3 dogs.
Dates found (year, month, day): [('2026', '05', '12'), ('2026', '05', '15')]


### Practice

Try modifying the examples above, or work on these:

In [3]:
sentences = [
    "She sells sea-shells by the sea shore.",
    "Call me at +1-800-555-0199 or email me at hello@example.com",
    "The price is $4.99, but with discount it's $3.50",
]

# 1. Extract all hyphenated words from sentences[0]
# Expected: ['sea-shells', 'sea-shore'] (approximately)
pattern_hyphen = r'...'  # <- your pattern here
print(re.findall(pattern_hyphen, sentences[0]))

# 2. Extract the phone number from sentences[1]
pattern_phone = r'...'  # <- your pattern here
print(re.findall(pattern_phone, sentences[1]))

# 3. Extract all prices (with $) from sentences[2]
pattern_price = r'...'  # <- your pattern here
print(re.findall(pattern_price, sentences[2]))

['She', ' se', 'lls', ' se', 'a-s', 'hel', 'ls ', 'by ', 'the', ' se', 'a s', 'hor']
['Cal', 'l m', 'e a', 't +', '1-8', '00-', '555', '-01', '99 ', 'or ', 'ema', 'il ', 'me ', 'at ', 'hel', 'lo@', 'exa', 'mpl', 'e.c']
['The', ' pr', 'ice', ' is', ' $4', '.99', ', b', 'ut ', 'wit', 'h d', 'isc', 'oun', 't i', "t's", ' $3', '.50']


## Word tokenization

### Space-based tokenization

Splitting on whitespace is the simplest approach — but it's naive. Punctuation sticks to words, contractions break unpredictably, and you get noisy tokens like `"father's"`, `"weep!"`, `"God,"`.

In [4]:
import re

# Read the entire Shakespeare text into a string
with open('../resources/shakespeare.txt', 'r') as f:
    raw = f.read()

# --- Method 1: split on whitespace (naive) ---
naive_tokens = raw.split()   # split() cuts on any whitespace
print("Naive split - total tokens:", len(naive_tokens))
print("Sample:", naive_tokens[300:308])

# Problem: punctuation sticks to words
# isalpha() returns False when a token contains digits or punctuation
noisy = []
for token in naive_tokens:
    if not token.isalpha():
        noisy.append(token)
print("\nTokens with punctuation/digits attached:", len(noisy))
print("Examples:", noisy[10:18])

# --- Method 2: regex tokenizer (better) ---
# [a-zA-Z]+ matches one or more letters
# (?:'[a-z]+)? optionally also matches contractions like "don't", "it's"
word_tokens = re.findall(r"[a-zA-Z]+(?:'[a-z]+)?", raw)
print("\nRegex tokenizer - total tokens:", len(word_tokens))
print("Sample:", word_tokens[300:308])

# Build the vocabulary: unique lowercased words
vocab = set()
for token in word_tokens:
    vocab.add(token.lower())
print("\nVocabulary size (unique words):", len(vocab))

Naive split - total tokens: 1068193
Sample: ['lack', 'of', 'work.', 'Would,', 'for', 'the', "King's", 'sake,']

Tokens with punctuation/digits attached: 281418
Examples: ['FLORENCE.', 'DIANA,', 'VIOLENTA,', 'MARIANA,', 'Lords,', 'Officers,', 'Soldiers,', 'etc.,']

Regex tokenizer - total tokens: 1073645
Sample: ['lack', 'of', 'work', 'Would', 'for', 'the', "King's", 'sake']

Vocabulary size (unique words): 26122


### Simple Tokenization in UNIX

#### The first step: tokenizing

`tr -sc 'A-Za-z' '\n'` — keep only letters, replace everything else with a newline. One word per line.

In [5]:
%%bash
tr -sc 'A-Za-z' '\n' < ../resources/shakespeare.txt | head -15


ALLS
WELL
THAT
ENDS
WELL
by
William
Shakespeare
Dramatis
Personae
KING
OF
FRANCE
THE


#### The second step: sorting

Pipe into `sort` — brings identical words together so we can count them.

In [6]:
%%bash
tr -sc 'A-Za-z' '\n' < ../resources/shakespeare.txt | sort | head -15


A
A
A
A
A
A
A
A
A
A
A
A
A
A


#### More counting

Add `uniq -c` to count consecutive duplicates, then `sort -rn` to rank by frequency.

In [7]:
%%bash
tr -sc 'A-Za-z' '\n' < ../resources/shakespeare.txt \
  | tr 'A-Z' 'a-z' \
  | sort \
  | uniq -c \
  | sort -rn \
  | head -20

33233 the
31284 and
27298 i
24291 to
20871 of
17579 a
16925 you
15216 my
13659 that
13192 in
11267 is
10405 d
10360 not
9423 with
9409 for
9387 it
9382 me
9378 s
8465 be
8304 his


### Tokenization in languages without spaces

In Chinese, Japanese, and Thai, words are **not separated by spaces**. Splitting on whitespace gives you nothing useful — you get full sentences as single "tokens". Segmentation requires dedicated algorithms or dictionaries.

| Language | Sentence | Space-split tokens |
|----------|----------|--------------------|
| English | `the cat sat` | `["the", "cat", "sat"]` ✓ |
| Chinese | `猫坐在垫子上` | `["猫坐在垫子上"]` ✗ |

In [8]:
en = "The cat sat on the mat"
zh = "猫坐在垫子上"          # "The cat sat on the mat" in Chinese

print("English — split():", en.split())
print(f"  → {len(en.split())} tokens\n")

print("Chinese — split():", zh.split())
print(f"  → {len(zh.split())} tokens  (whole sentence = one 'token'!)")
print()

# Chinese characters are already word-like units — iterate over chars as a proxy
print("Chinese — character-level (rough baseline):", list(zh))
print(f"  → {len(zh)} characters")

English — split(): ['The', 'cat', 'sat', 'on', 'the', 'mat']
  → 6 tokens

Chinese — split(): ['猫坐在垫子上']
  → 1 tokens  (whole sentence = one 'token'!)

Chinese — character-level (rough baseline): ['猫', '坐', '在', '垫', '子', '上']
  → 6 characters


### Practice

In [9]:
import re
from collections import Counter

# Read the Shakespeare file
with open('../resources/shakespeare.txt', 'r') as f:
    raw = f.read()

# Helper function: prints a tick if done, reminder if not
def check(label, value):
    if value is ...:
        print("[ ]", label, "- not filled in yet")
    else:
        print("[✓]", label, ":", value)


# --- Exercise 1 ---
# Get all word tokens using the regex from above,
# then build a set of unique lowercased words (the vocabulary).
tokens = ...  # Hint: re.findall(r"[a-zA-Z]+(?:'[a-z]+)?", raw)
vocab  = ...  # Hint: a set() of lowercased tokens
check("Unique words", len(vocab) if vocab is not ... else ...)


# --- Exercise 2 ---
# Find the 10 most common words.
# Counter(list) counts how often each item appears.
# .most_common(10) returns the top 10 as [(word, count), ...]
most_common = ...  # Hint: Counter(tokens).most_common(10)
check("Top 10 words", most_common)


# --- Exercise 3 ---
# Count hapax legomena: words that appear exactly ONCE.
# These are very rare words — a classic linguistic measure.
# Hint: use Counter(tokens), then loop and keep words with count == 1
hapax = ...  # should be a list or set of words
check("Hapax legomena", len(hapax) if hapax is not ... else ...)


# --- Exercise 4 ---
# Type-Token Ratio = unique words / total tokens
# A value closer to 1.0 means very varied vocabulary.
ttr = ...  # Hint: len(vocab) / len(tokens)
check("TTR", round(ttr, 4) if ttr is not ... else ...)

[ ] Unique words - not filled in yet
[ ] Top 10 words - not filled in yet
[ ] Hapax legomena - not filled in yet
[ ] TTR - not filled in yet


## Byte Pair Encoding

Instead of splitting on spaces or individual characters, **BPE lets the data decide** how to tokenize. The result is *subword* tokens — whole words when they're frequent, character chunks when they're not.

| Approach | "newer" | "lower" (unseen) |
|----------|---------|-----------------|
| Whitespace | `newer` | `lower` ✗ OOV |
| Char-level | `n e w e r` | `l o w e r` |
| **BPE** | `newer_` | `low` + `er_` ✓ |

Two parts:
- **Token learner** — scans a training corpus and learns a vocabulary of merge rules
- **Token segmenter** — applies those rules (greedily, in learned order) to new text

### BPE token learner algorithm

In [10]:
# Corpus from the slides
corpus = "low low low low low lowest lowest newer newer newer newer newer newer wider wider wider new new"

# --- Step 1: Count how often each word appears ---
word_counts = {}
for word in corpus.split():
    if word not in word_counts:
        word_counts[word] = 0
    word_counts[word] += 1

# --- Step 2: Build the initial character vocabulary ---
# Split each word into individual characters and add '_' as end-of-word marker
# Example: "low" → "l o w _"
vocab = {}
for word, count in word_counts.items():
    chars = list(word) + ['_']    # ['l', 'o', 'w', '_']
    key   = ' '.join(chars)        # "l o w _"
    vocab[key] = count

print("Initial character vocabulary:")
for word, count in vocab.items():
    print(" ", count, "x", word)


# --- Helper: count how often each adjacent pair appears ---
def count_pairs(vocab):
    pairs = {}
    for word, freq in vocab.items():
        symbols = word.split()                       # ["l", "o", "w", "_"]
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i+1])        # e.g. ("l", "o")
            if pair not in pairs:
                pairs[pair] = 0
            pairs[pair] += freq
    return pairs


# --- Helper: merge the best pair everywhere in the vocab ---
def merge_pair(best_pair, vocab):
    new_vocab = {}
    old = best_pair[0] + ' ' + best_pair[1]    # "e r"
    new = best_pair[0] + best_pair[1]           # "er"
    for word, count in vocab.items():
        new_word = word.replace(old, new)        # replace every occurrence
        new_vocab[new_word] = count
    return new_vocab


# --- Run BPE: learn k merges ---
k      = 10
merges = []    # we store the merge rules here for the segmenter

print("\nLearning merges:")
for i in range(k):
    pairs     = count_pairs(vocab)
    best_pair = max(pairs, key=pairs.get)      # pick the most frequent pair
    vocab     = merge_pair(best_pair, vocab)
    merges.append(best_pair)
    merged = best_pair[0] + best_pair[1]
    print("Merge", i+1, ":", best_pair[0], "+", best_pair[1], "→", merged, " (count =", pairs[best_pair], ")")

print("\nFinal vocabulary:")
for word, count in vocab.items():
    print(" ", count, "x", word)

Initial character vocabulary:
  5 x l o w _
  2 x l o w e s t _
  6 x n e w e r _
  3 x w i d e r _
  2 x n e w _

Learning merges:
Merge 1 : e + r → er  (count = 9 )
Merge 2 : er + _ → er_  (count = 9 )
Merge 3 : n + e → ne  (count = 8 )
Merge 4 : ne + w → new  (count = 8 )
Merge 5 : l + o → lo  (count = 7 )
Merge 6 : lo + w → low  (count = 7 )
Merge 7 : new + er_ → newer_  (count = 6 )
Merge 8 : low + _ → low_  (count = 5 )
Merge 9 : w + i → wi  (count = 3 )
Merge 10 : wi + d → wid  (count = 3 )

Final vocabulary:
  5 x low_
  2 x low e s t _
  6 x newer_
  3 x wid er_
  2 x new _


### BPE token segmenter algorithm

Apply the learned merges **greedily and in order** to any new word. Training frequencies don't matter here — only the merge sequence does.

- `newer` → seen in training → becomes one token `newer_`
- `lower` → unseen → partially matches → splits into `low` + `er_`

In [11]:
def segment(word, merges):
    # Start by splitting the word into individual characters + end marker
    # "newer" → ['n', 'e', 'w', 'e', 'r', '_']
    symbols = list(word) + ['_']

    # Apply each merge rule in the order it was learned
    for pair in merges:
        i = 0
        while i < len(symbols) - 1:
            # Check if the current and next symbol match this merge rule
            if symbols[i] == pair[0] and symbols[i+1] == pair[1]:
                merged  = pair[0] + pair[1]                     # e.g. "er"
                symbols = symbols[:i] + [merged] + symbols[i+2:] # replace the two with one
                # don't advance i — the new merged symbol might match the next rule
            else:
                i += 1

    return symbols


# Test on words from the slides
test_words = ["newer", "lower", "lowest", "widest", "newish"]

print("Word          Tokens")
print("-" * 40)
for word in test_words:
    tokens = segment(word, merges)
    print(word.ljust(14), tokens)

Word          Tokens
----------------------------------------
newer          ['newer_']
lower          ['low', 'er_']
lowest         ['low', 'e', 's', 't', '_']
widest         ['wid', 'e', 's', 't', '_']
newish         ['new', 'i', 's', 'h', '_']


### Practice

In [12]:
# --- Exercise 1 ---
# Change k to 20 in the learner cell above and re-run it.
# At which merge number does "low" first appear as a single token?
# Write your answer as a comment:

# Answer: merge number ___ produces "low"


# --- Exercise 2 ---
# Add "slowest" three times to the corpus and re-train BPE.
# Does the merge order change compared to before?

new_corpus = corpus + " slowest slowest slowest"

# Build vocabulary for the new corpus (same steps as before)
new_word_counts = {}
for word in new_corpus.split():
    if word not in new_word_counts:
        new_word_counts[word] = 0
    new_word_counts[word] += 1

new_vocab = {}
for word, count in new_word_counts.items():
    chars = list(word) + ['_']
    key   = ' '.join(chars)
    new_vocab[key] = count

# Run BPE on the new vocab — copy the loop from above and paste here
# ... your code here


# --- Exercise 3 ---
# Segment these words using the merges learned above.
# Which come out as a single token? Which are split?
for word in ["lowest", "newest", "slower"]:
    print(word, "→", segment(word, merges))

lowest → ['low', 'e', 's', 't', '_']
newest → ['new', 'e', 's', 't', '_']
slower → ['s', 'low', 'er_']


## Word Normalization and other issues

### Word Normalization

Converting tokens into a **standard form** before processing. Common tasks:

| Task | Before | After |
|------|--------|-------|
| Lowercase | `Fed` | `fed` |
| Remove abbrev. dots | `U.S.A.` | `USA` |
| Collapse repeated chars | `loooove` | `loove` |
| Normalize numbers | `$4.99` | keep or strip |

In [13]:
import re

# --- Remove periods from abbreviations ---
abbrevs = ["U.S.A.", "Ph.D.", "m.p.h.", "e.g."]
print("Abbreviation normalization:")
for w in abbrevs:
    print(" ", w, "→", w.replace('.', ''))

# --- Collapse repeated characters ---
# Regex: (.) captures any char, \1{2,} means 2+ more of the same
# Replace with just 2 of that character
text = "I loooooove this! Sooooo gooood!"
normalized = re.sub(r'(.)\1{2,}', r'\1\1', text)
print("\nCollapse repeated chars:")
print("  Before:", text)
print("  After: ", normalized)

Abbreviation normalization:
  U.S.A. → USA
  Ph.D. → PhD
  m.p.h. → mph
  e.g. → eg

Collapse repeated chars:
  Before: I loooooove this! Sooooo gooood!
  After:  I loove this! Soo good!


### Case folding

In [14]:
sentence = "The FBI investigated General Motors and Apple Inc."

# Lowercasing helps search: "Apple" and "apple" match the same query
print("Original :", sentence)
print("Lowercased:", sentence.lower())

# But case sometimes carries meaning — lowercasing loses that
print()
pairs = [
    ("US",      "country name"),
    ("us",      "pronoun"),
    ("Apple",   "the company"),
    ("apple",   "the fruit"),
    ("General", "military rank"),
    ("general", "adjective"),
]
for word, meaning in pairs:
    print(" ", word.ljust(10), "=", meaning, " → lower:", word.lower())

Original : The FBI investigated General Motors and Apple Inc.
Lowercased: the fbi investigated general motors and apple inc.

  US         = country name  → lower: us
  us         = pronoun  → lower: us
  Apple      = the company  → lower: apple
  apple      = the fruit  → lower: apple
  General    = military rank  → lower: general
  general    = adjective  → lower: general


### Lemmatization

In [15]:
# Lemmatization maps words to their dictionary base form (the "lemma")
# am / is / are / was / were  →  be
# cats / cat's              →  cat
# running / ran             →  run

# A small manual lemma dictionary
lemma_dict = {
    "am":       "be",
    "is":       "be",
    "are":      "be",
    "was":      "be",
    "were":     "be",
    "cats":     "cat",
    "running":  "run",
    "ran":      "run",
    "better":   "good",
    "best":     "good",
    "studies":  "study",
}

sentence = "The cats are running and she was better than before"

words      = sentence.split()
lemmatized = []
for word in words:
    lower_word = word.lower()
    # If we know the lemma, use it — otherwise keep the word as-is
    lemma = lemma_dict.get(lower_word, lower_word)
    lemmatized.append(lemma)

print("Original :", sentence)
print("Lemmatized:", ' '.join(lemmatized))
print()
print("Note: real lemmatizers (like spaCy) use large dictionaries")
print("and part-of-speech info to handle thousands of word forms.")

Original : The cats are running and she was better than before
Lemmatized: the cat be run and she be good than before

Note: real lemmatizers (like spaCy) use large dictionaries
and part-of-speech info to handle thousands of word forms.


### Stemming

In [16]:
# Stemming: crudely chop off suffixes — faster but less accurate than lemmatization
# It doesn't use a dictionary, just rules.

def simple_stem(word):
    word = word.lower()
    # Rules are checked in order — the first match wins
    if word.endswith("tion"):
        return word[:-4]
    if word.endswith("ness"):
        return word[:-4]
    if word.endswith("ing"):
        return word[:-3]
    if word.endswith("est"):
        return word[:-3]
    if word.endswith("ed"):
        return word[:-2]
    if word.endswith("ly"):
        return word[:-2]
    if word.endswith("er"):
        return word[:-2]
    if word.endswith("s") and not word.endswith("ss"):
        return word[:-1]
    return word

test_words = ["running", "cats", "happily", "fastest", "computed", "connection"]

print("Word            Stem")
print("-" * 30)
for word in test_words:
    print(word.ljust(16), simple_stem(word))

Word            Stem
------------------------------
running          runn
cats             cat
happily          happi
fastest          fast
computed         comput
connection       connec


### Porter Stemmer

In [17]:
# The Porter Stemmer runs rules in a cascade (output of step 1 feeds into step 2, etc.)
# Here is Step 1a — just the plural/verb suffix rules:

def porter_step1a(word):
    word = word.lower()

    if word.endswith("sses"):   # caresses → caress
        return word[:-2]
    if word.endswith("ies"):    # ponies → poni
        return word[:-2]
    if word.endswith("ss"):     # caress → caress (leave alone)
        return word
    if word.endswith("s"):      # cats → cat
        return word[:-1]
    return word

examples = ["caresses", "ponies", "caress", "cats", "buses", "happiness"]

print("Word            After Step 1a")
print("-" * 35)
for word in examples:
    print(word.ljust(16), porter_step1a(word))

print()
print("The full Porter Stemmer has 5 steps and ~60 rules.")
print("Python's NLTK library has a complete built-in implementation:")

Word            After Step 1a
-----------------------------------
caresses         caress
ponies           poni
caress           caress
cats             cat
buses            buse
happiness        happiness

The full Porter Stemmer has 5 steps and ~60 rules.
Python's NLTK library has a complete built-in implementation:


### Sentence Segmentation

In [18]:
import re

text = "Dr. Smith works at Apple Inc. in Palo Alto. She earns $2.5M per year. Is that a lot? Yes!"

# --- Naive: split on every . ! ? ---
naive = re.split(r'[.!?]', text)
print("Naive split (wrong):")
for s in naive:
    s = s.strip()
    if s:
        print(" ", repr(s))

# Problem: "Dr.", "Inc.", and "2.5" all trigger a split — false positives!

# --- Better: split only when period is followed by space + capital letter ---
# and the word before it is not a known abbreviation
abbreviations = {"dr", "mr", "mrs", "ms", "prof", "inc", "ltd", "vs", "etc"}

def split_sentences(text):
    sentences = []
    current   = ""
    words     = text.split()

    for word in words:
        current += word + " "
        # A sentence ends if this word ends with . ! or ?
        if word[-1] in '.!?':
            clean = word.rstrip('.!?').lower()    # remove punctuation, lowercase
            if clean not in abbreviations:         # skip known abbreviations
                sentences.append(current.strip())
                current = ""

    if current.strip():          # don't forget the last sentence
        sentences.append(current.strip())
    return sentences

print("\nBetter split:")
for s in split_sentences(text):
    print(" ", repr(s))

Naive split (wrong):
  'Dr'
  'Smith works at Apple Inc'
  'in Palo Alto'
  'She earns $2'
  '5M per year'
  'Is that a lot'
  'Yes'

Better split:
  'Dr. Smith works at Apple Inc. in Palo Alto.'
  'She earns $2.5M per year.'
  'Is that a lot?'
  'Yes!'


### Practice

In [19]:
def check(label, value):
    if value is ...:
        print("[ ]", label, "- not filled in yet")
    else:
        print("[✓]", label, ":", value)


# --- Exercise 1 ---
# Normalize this tweet: lowercase it and collapse repeated characters.
tweet = "I LOOOOVE Computational Linguistics sooooo much!!!"

normalized = ...  # your code here
check("Normalized tweet", normalized)


# --- Exercise 2 ---
# Use simple_stem() from above on these words.
# Which ones get over-stemmed (the result is not a real word)?
words = ["caring", "universal", "generously", "happiness", "running"]

stems = []
for word in words:
    stems.append(simple_stem(word))   # simple_stem is defined in the Stemming cell above
check("Stems", stems)


# --- Exercise 3 ---
# Add "PhD" and "Fig" to the abbreviations set in split_sentences()
# and re-run it on this text. How many sentences does it find?
text2 = "Prof. Lee published in Fig. 3 of the paper. It was cited 100 times. Amazing!"

# Hint: copy split_sentences() here, add "phd" and "fig" to abbreviations,
# then call it on text2
sentences = ...  # your code here
check("Sentence count", len(sentences) if sentences is not ... else ...)

[ ] Normalized tweet - not filled in yet
[✓] Stems : ['car', 'universal', 'generous', 'happi', 'runn']
[ ] Sentence count - not filled in yet
